# 01 — The ML Data Lifecycle

**Difficulty:** ⭐⭐ | **Estimated Time:** 2 hours  
**Week 4 · Data Fundamentals & Preprocessing Pipelines**

---

## Why Does This Matter for ML?

Every ML model is only as good as the data it was trained on. Before you write a single line of model code, the data has already traveled through a long journey — collected from somewhere, labeled by someone, stored in some format, versioned (or not), and processed in some way. If any stage of that journey goes wrong, your model will fail — silently or loudly.

In real ML teams, **data engineering and data pipeline work consumes roughly 80% of the project time**. Understanding the full lifecycle helps you:
- Debug why your model's accuracy suddenly dropped
- Reproduce a result from 6 months ago
- Avoid legal and ethical pitfalls (biased sampling, poor labeling)
- Collaborate with data engineers and domain experts

---

## Real-World Analogy: Farm to Table

Think of building an ML model like running a restaurant. Your **data is the food**.

| ML Stage | Restaurant Analogy |
|---|---|
| Data Collection | Planting seeds & harvesting vegetables |
| Data Labeling | Inspecting each vegetable — is it ripe? rotten? |
| Data Storage | Organizing the walk-in fridge with clear labels |
| Data Versioning | Keeping a recipe book that records every batch |
| Data Processing | Washing, chopping, cooking before serving |

A Michelin-star chef cannot make great food from rotten vegetables, no matter how skilled they are. Similarly, no ML algorithm can produce reliable predictions from bad data.

---

**Our running theme:** Credit card fraud detection — a real, high-stakes binary classification problem with messy, imbalanced, noisy data.

## Setup: Import Libraries

Before anything else, we load the tools we'll need throughout this notebook.

In [ ]:
# Standard data science imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
import seaborn as sns

# Built-in Python: for file I/O, database simulation, and timestamps
import sqlite3          # lightweight SQL database built into Python — no server needed
import os               # file system operations
import json             # reading/writing JSON format
import datetime         # working with dates and timestamps
import warnings
warnings.filterwarnings('ignore')  # suppress minor warnings for cleaner output

# Set a consistent visual style for all plots
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

# Fix the random seed so results are reproducible every time you run the notebook
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('Libraries loaded successfully.')
print(f'NumPy {np.__version__} | Pandas {pd.__version__}')

---

## Stage 1 — Data Collection

### What Is It?
Data collection is the process of gathering raw observations from the world. Before any ML can happen, you need data. For a fraud detection system, this means capturing every transaction that flows through the payment network.

### Where Does Fraud Detection Data Come From?
| Source | What It Captures |
|---|---|
| Transaction logs (SQL database) | Amount, merchant, timestamp, card number |
| User behavior APIs | Login patterns, typical spending hours |
| Device fingerprints | IP address, browser, GPS location |
| Manual reports | Customer calls saying "I didn't do this" |

### The Big Questions Before Collecting
1. **Is it representative?** If you only collect data from US users, your model will fail on European transactions.
2. **Is it fresh?** A dataset from 2019 may miss fraud patterns that emerged during COVID.
3. **Is it biased?** If fraud investigators historically checked more transactions from young users, the training data will have more fraud labels for young users — not because they commit more fraud, but because they were watched more.

### Sampling Bias — A Critical Danger
**Plain English:** Sampling bias means your data does not fairly represent reality. It's like polling only wealthy neighborhoods about poverty — the answers will be wrong because you're not asking the right people.

In fraud detection: if your bank only investigated transactions > \$500, you have no labels for small-amount fraud. Your model learns that small transactions are always legitimate — which is exactly what sophisticated fraudsters exploit.

In [ ]:
# ============================================================
# GENERATE SYNTHETIC CREDIT CARD TRANSACTION DATASET
# ============================================================
# We simulate 10,000 credit card transactions with realistic patterns.
# Real-world fraud rates are typically 0.1%–3%; we use ~2% here.

N = 10_000   # total number of transactions
FRAUD_RATE = 0.02  # 2% of transactions are fraudulent

np.random.seed(RANDOM_SEED)

# --- Transaction IDs: unique identifier for each transaction ---
transaction_ids = [f'TXN{str(i).zfill(6)}' for i in range(1, N + 1)]  # e.g., TXN000001

# --- Timestamps: spread over 1 year (365 days) ---
# We generate random seconds since a base date and convert to datetime
base_date = datetime.datetime(2023, 1, 1)
random_seconds = np.random.randint(0, 365 * 24 * 3600, size=N)  # random second within the year
timestamps = [base_date + datetime.timedelta(seconds=int(s)) for s in random_seconds]

# --- Transaction amounts: log-normal distribution mimics real spending ---
# Most purchases are small ($10-$100); some are large; fraud can be any amount
amounts = np.round(np.random.lognormal(mean=4.0, sigma=1.2, size=N), 2)  # log-normal in dollars
amounts = np.clip(amounts, 1.0, 15000.0)  # cap at $15,000 to avoid extreme outliers

# --- Merchant categories: discrete categories weighted by real-world frequency ---
categories = ['grocery', 'restaurant', 'gas_station', 'online_retail',
               'travel', 'entertainment', 'healthcare', 'atm_withdrawal']
# Realistic weights: groceries and restaurants are most common
cat_weights = [0.25, 0.20, 0.15, 0.18, 0.08, 0.06, 0.05, 0.03]
merchant_categories = np.random.choice(categories, size=N, p=cat_weights)

# --- User IDs: 500 unique users making ~20 transactions each on average ---
user_ids = [f'USR{str(i).zfill(4)}' for i in np.random.randint(1, 501, size=N)]

# --- Fraud labels: binary 0/1, with ~2% fraud rate ---
# Use Bernoulli random variable: each transaction independently has 2% chance of fraud
is_fraud = np.random.binomial(n=1, p=FRAUD_RATE, size=N)  # Bernoulli(0.02)

# --- Assemble into a DataFrame ---
df = pd.DataFrame({
    'transaction_id': transaction_ids,
    'timestamp':      timestamps,
    'amount':         amounts,
    'merchant_category': merchant_categories,
    'user_id':        user_ids,
    'is_fraud':       is_fraud
})

print(f'Dataset shape: {df.shape}  ({N:,} rows × {df.shape[1]} columns)')
print(f'Fraud transactions: {is_fraud.sum():,}  ({is_fraud.mean()*100:.1f}%)')
print(f'Legit transactions: {(1 - is_fraud).sum():,}  ({(1-is_fraud.mean())*100:.1f}%)')
df.head()

---

## Stage 2 — Data Labeling

### What Is It?
**Plain English:** A label is the answer to the question your model is trying to learn. For fraud detection, the label is `is_fraud` — did this transaction turn out to be fraudulent?

Supervised ML (the most common type) requires labeled examples: the algorithm learns by seeing thousands of examples where the correct answer is already known.

### Who Does the Labeling?
| Method | Example | Trade-off |
|---|---|---|
| **Domain experts** | Bank fraud analysts review flagged transactions | High quality, very expensive |
| **Crowdsourcing** | Amazon Mechanical Turk workers | Cheap, variable quality |
| **Programmatic (weak supervision)** | "Any charge > $5,000 at 3am is fraud" | Fast, noisy labels |
| **Customer reports** | Card holder calls to report fraud | Reliable but delayed, underreports |

### The Cost of Labeling
If you have 10,000 fraud transactions and hire experts at $2 per label:
- **10,000 × $2 = $20,000** just for fraud labels
- At scale (1 million transactions): $2,000,000
- This is why **label efficiency** (getting maximum information from minimum labels) is an active research area.

### Label Noise — A Hidden Problem
**Plain English:** Label noise means some of your labels are wrong. Even experts disagree — two fraud analysts might classify the same transaction differently. If 5% of your labels are flipped (fraud marked as legitimate and vice versa), your model learns incorrect patterns.

In [ ]:
# ============================================================
# LABEL NOISE: Simulate 5% label flipping and measure impact
# ============================================================
# In real labeling pipelines, annotators make mistakes.
# We'll randomly flip 5% of labels and see how it changes the dataset.

NOISE_RATE = 0.05  # 5% of labels will be incorrectly flipped

# Create a copy so we preserve the original clean labels
df_noisy = df.copy()

# Randomly select 5% of rows to flip their label
# np.random.choice selects indices WITHOUT replacement
n_noisy = int(N * NOISE_RATE)  # number of labels to flip = 500
noisy_indices = np.random.choice(df_noisy.index, size=n_noisy, replace=False)

# Flip the label: 0 becomes 1, 1 becomes 0
# Using XOR (^) with 1 flips a binary value: 0^1=1, 1^1=0
df_noisy.loc[noisy_indices, 'is_fraud'] = df_noisy.loc[noisy_indices, 'is_fraud'] ^ 1

# --- Compare class distributions before and after noise injection ---
original_fraud_rate  = df['is_fraud'].mean() * 100
noisy_fraud_rate     = df_noisy['is_fraud'].mean() * 100

# Count how many labels were flipped in each direction
original_labels = df['is_fraud'].values
noisy_labels    = df_noisy['is_fraud'].values
flipped_to_fraud  = ((original_labels == 0) & (noisy_labels == 1)).sum()  # legit → fraud
flipped_to_legit  = ((original_labels == 1) & (noisy_labels == 0)).sum()  # fraud → legit

print('=== LABEL NOISE ANALYSIS ===')
print(f'Original fraud rate :  {original_fraud_rate:.2f}%  ({df["is_fraud"].sum()} transactions)')
print(f'Noisy fraud rate    :  {noisy_fraud_rate:.2f}%  ({df_noisy["is_fraud"].sum()} transactions)')
print(f'Labels flipped (legit → fraud): {flipped_to_fraud}')
print(f'Labels flipped (fraud → legit): {flipped_to_legit}')
print(f'\nImpact: The noisy dataset has {abs(df_noisy["is_fraud"].sum() - df["is_fraud"].sum())} '
      f'more/fewer fraud labels than reality.')
print('A model trained on this will have systematically wrong beliefs about what fraud looks like.')

In [ ]:
# ============================================================
# VISUALIZE: Label distribution before vs after noise
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Effect of 5% Label Noise on Class Distribution', fontsize=14, fontweight='bold')

# Define colors: blue for legit, red for fraud
colors = ['#4C9BE8', '#E85D5D']

for ax, (title, data) in zip(axes, [
    ('BEFORE Noise (Ground Truth)', df['is_fraud']),
    ('AFTER 5% Label Noise',        df_noisy['is_fraud'])
]):
    counts = data.value_counts().sort_index()  # [legit_count, fraud_count]
    bars = ax.bar(['Legitimate (0)', 'Fraud (1)'], counts.values, color=colors, edgecolor='white', linewidth=1.5)
    
    # Annotate each bar with its count and percentage
    for bar, count in zip(bars, counts.values):
        pct = count / len(data) * 100
        ax.text(bar.get_x() + bar.get_width()/2,  # x position: center of bar
                bar.get_height() + 30,             # y position: just above bar
                f'{count:,}\n({pct:.1f}%)',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    ax.set_title(title, fontsize=12)
    ax.set_ylabel('Number of Transactions')
    ax.set_ylim(0, N * 1.12)  # extra headroom for annotations
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))  # add commas

plt.tight_layout()
plt.show()

# Key insight: even a 5% noise rate significantly shifts the apparent fraud rate,
# causing a model to be either over-sensitive or under-sensitive to fraud.

---

## Stage 3 — Data Storage

### What Is It?
Once data is collected and labeled, it needs to be stored somewhere. The choice of storage format affects how fast you can read it, whether the schema is enforced, and how efficiently it compresses.

### Common Formats Compared

| Format | Human Readable | Schema Enforced | Compression | ML Use Case |
|---|---|---|---|---|
| **CSV** | Yes | No | None | Quick prototyping, small data |
| **JSON** | Yes | No | None | APIs, nested/hierarchical data |
| **SQLite/SQL** | Via query | Yes (types, constraints) | Moderate | Structured relational data |
| **Parquet** | No | Yes (schema in file) | Excellent (~10x) | Production ML, large datasets |

### Why Does Format Matter?
**Plain English:** Imagine storing 1 million transactions. As a CSV, that might be 500 MB. As Parquet, it might compress to 50 MB and load 5–10x faster because Parquet only reads the columns you ask for (columnar storage), while CSV reads every row in full even if you only need 2 columns.

For fraud detection in production:
- Raw transaction logs → SQL database (for querying by user, date, amount)
- Feature-engineered training datasets → Parquet (fast loading for model training)
- Experiment configs and metadata → JSON

In [ ]:
# ============================================================
# DATA STORAGE: CSV, SQLite, and Parquet (in-memory comparison)
# ============================================================
import io      # in-memory byte streams (lets us simulate files without touching disk)
import time    # for measuring read/write speed

# ---- 1. CSV: Save to in-memory string buffer (no actual file created) ----
csv_buffer = io.StringIO()  # StringIO acts like a file but lives in RAM
df.to_csv(csv_buffer, index=False)
csv_size_kb = csv_buffer.tell() / 1024  # .tell() returns current position = total bytes written

# Read back from the CSV buffer
csv_buffer.seek(0)  # rewind to the beginning before reading
t0 = time.time()
df_from_csv = pd.read_csv(csv_buffer)
csv_read_ms = (time.time() - t0) * 1000  # convert seconds to milliseconds

# ---- 2. SQLite: In-memory database using Python's built-in sqlite3 ----
# ':memory:' tells SQLite to create the database entirely in RAM (no file on disk)
conn = sqlite3.connect(':memory:')

# Write our DataFrame into a SQL table called 'transactions'
# if_exists='replace' drops and recreates the table if it already exists
df.to_sql('transactions', conn, if_exists='replace', index=False)

# Query: read back only fraud transactions using SQL syntax
t0 = time.time()
df_from_sql = pd.read_sql(
    'SELECT * FROM transactions WHERE is_fraud = 1',  # SQL filter for fraud only
    conn
)
sql_read_ms = (time.time() - t0) * 1000

# ---- 3. Parquet: Columnar binary format (requires pyarrow or fastparquet) ----
parquet_buffer = io.BytesIO()  # Parquet is binary, so we use BytesIO (not StringIO)
try:
    df.to_parquet(parquet_buffer, index=False)
    parquet_size_kb = parquet_buffer.tell() / 1024
    parquet_buffer.seek(0)
    t0 = time.time()
    df_from_parquet = pd.read_parquet(parquet_buffer)
    parquet_read_ms = (time.time() - t0) * 1000
    parquet_available = True
except Exception as e:
    parquet_available = False
    print(f'Parquet not available (install pyarrow): {e}')
    parquet_size_kb, parquet_read_ms = None, None

# ---- Summary comparison ----
print('=== STORAGE FORMAT COMPARISON ===')
print(f'{"Format":<12} {"Size (KB)":>12} {"Read Time (ms)":>16}')
print('-' * 42)
print(f'{"CSV":<12} {csv_size_kb:>12.1f} {csv_read_ms:>16.1f}')
if parquet_available:
    print(f'{"Parquet":<12} {parquet_size_kb:>12.1f} {parquet_read_ms:>16.1f}')
    print(f'\nParquet compression ratio vs CSV: {csv_size_kb/parquet_size_kb:.1f}x smaller')
print(f'\nSQL query (fraud only) returned {len(df_from_sql)} rows in {sql_read_ms:.1f} ms')
print('SQL lets you filter BEFORE loading — you only pull what you need.')

In [ ]:
# ============================================================
# SQLite DEMO: Richer queries on our transaction database
# ============================================================
# SQL is the universal language of structured data.
# Here we demonstrate typical analytical queries a fraud analyst would run.

# Query 1: Fraud rate by merchant category
# GROUP BY splits rows into groups; AVG(is_fraud) gives the fraud rate per group
fraud_by_category = pd.read_sql("""
    SELECT 
        merchant_category,
        COUNT(*) AS total_transactions,
        SUM(is_fraud) AS fraud_count,
        ROUND(AVG(is_fraud) * 100, 2) AS fraud_rate_pct,
        ROUND(AVG(amount), 2) AS avg_amount
    FROM transactions
    GROUP BY merchant_category
    ORDER BY fraud_rate_pct DESC
""", conn)

print('=== FRAUD RATE BY MERCHANT CATEGORY ===')
print(fraud_by_category.to_string(index=False))

print('\n=== TOTAL FRAUD LOSSES ===')
# Query 2: Total dollar amount lost to fraud per category
losses = pd.read_sql("""
    SELECT 
        merchant_category,
        ROUND(SUM(CASE WHEN is_fraud = 1 THEN amount ELSE 0 END), 2) AS fraud_losses_usd
    FROM transactions
    GROUP BY merchant_category
    ORDER BY fraud_losses_usd DESC
    LIMIT 5
""", conn)
print(losses.to_string(index=False))

---

## Stage 4 — Data Versioning

### What Is It?
**Plain English:** Imagine you trained a fraud detection model in January, and it performed great. By July, the accuracy has dropped significantly. To debug this, you need to go back to the exact dataset used in January — but if you've been overwriting your data files, that's impossible.

**Data versioning** means keeping snapshots of your datasets over time, just like Git keeps snapshots of your code.

### Git for Code → DVC for Data
- **Git** tracks changes to text files (your Python scripts, configs)
- **DVC (Data Version Control)** tracks changes to large data files (CSVs, models, datasets)
- DVC stores data in cloud storage (S3, GCS) and tracks pointers in Git

### The Golden Rule of Data Versioning
> **Never overwrite raw data. Always create new versions.**

Raw data is your source of truth. If you overwrite it with preprocessed data, you can never go back. It's like a scientist erasing their original experiment notes — the record is lost forever.

### Best Practices
1. Append a timestamp or version number to every dataset filename
2. Keep a **data lineage log** — a record of every transformation applied
3. Store metadata alongside data (who created it, from what source, when)
4. Use immutable (read-only) permissions on raw data files

In [ ]:
# ============================================================
# DATA VERSIONING: Create versioned snapshots with metadata
# ============================================================
# We simulate a real versioning workflow:
#   v1 = raw transactions (as collected)
#   v2 = transactions after adding a new engineered feature

# --- Version 1: Raw data with creation timestamp ---
df_v1 = df.copy()
# Add a metadata column recording when this version was created
df_v1['_version'] = 'v1'
df_v1['_created_at'] = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
df_v1['_source'] = 'raw_transaction_log'  # provenance: where did this data come from?

# --- Version 2: Add a feature — hour of day (useful for fraud detection) ---
# Fraudsters often strike at unusual hours (late night, early morning)
df_v2 = df.copy()
df_v2['hour_of_day'] = pd.to_datetime(df_v2['timestamp']).dt.hour  # extract hour (0-23)
df_v2['is_weekend'] = pd.to_datetime(df_v2['timestamp']).dt.dayofweek >= 5  # Sat/Sun = True
df_v2['_version'] = 'v2'
df_v2['_created_at'] = (datetime.datetime.now() + datetime.timedelta(days=30)).strftime('%Y-%m-%d %H:%M:%S')
df_v2['_source'] = 'feature_engineering_v1'  # derived from v1, not raw

# --- Data Lineage Log: track every transformation ---
# In production systems, this log is stored in a database or MLflow tracking server
lineage_log = []

def log_transformation(version, description, rows_in, rows_out, columns_added=None):
    """Record a data transformation step in the lineage log."""
    entry = {
        'timestamp':       datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'version':         version,
        'description':     description,
        'rows_in':         rows_in,
        'rows_out':        rows_out,
        'rows_changed':    rows_in - rows_out,  # how many rows were filtered/removed
        'columns_added':   columns_added or []  # new columns created in this step
    }
    lineage_log.append(entry)
    return entry

# Log the transformations we performed
log_transformation('v1 → created', 'Raw data ingested from transaction log', 0, N)
log_transformation('v1 → v2', 'Feature engineering: added hour_of_day, is_weekend', N, N,
                   columns_added=['hour_of_day', 'is_weekend'])

print('=== DATA LINEAGE LOG ===')
for i, entry in enumerate(lineage_log, 1):
    print(f'\nStep {i}: [{entry["timestamp"]}]')
    print(f'  Transform: {entry["description"]}')
    print(f'  Rows: {entry["rows_in"]:,} → {entry["rows_out"]:,}  (Δ {entry["rows_changed"]:,})')
    if entry['columns_added']:
        print(f'  New columns: {entry["columns_added"]}')

print(f'\nv1 shape: {df_v1.shape}')
print(f'v2 shape: {df_v2.shape}')
print(f'New columns in v2: {[c for c in df_v2.columns if c not in df_v1.columns]}')

---

## Stage 5 — Data Processing

### What Is It?
**Plain English:** Raw data is almost never in the right shape for an ML model. Data processing transforms raw observations into a clean, structured, numeric format that algorithms can actually learn from.

### The 80/20 Rule of ML Projects
> In practice, **80% of an ML engineer's time** is spent on data collection, cleaning, and processing. Only **20% is the actual modeling**.

This is counterintuitive to beginners who think ML is mostly about choosing algorithms. The hard part is getting good data.

### The Processing Pipeline for Fraud Detection
```
Raw Transactions
      ↓
 Handle Missing Values   ← (covered in notebook 2)
      ↓
 Fix Inconsistencies     ← (covered in notebook 2)
      ↓
 Feature Engineering     ← (covered in notebooks 3–5)
      ↓
 Encoding Categoricals   ← (covered in notebook 6)
      ↓
 Scaling/Normalization   ← (covered in notebook 7)
      ↓
 Train/Test Split        ← (covered in notebook 8)
      ↓
   Model Input
```

This entire week's curriculum covers the processing stage in depth. For now, we demonstrate the concept with a simple example.

In [ ]:
# ============================================================
# DATA PROCESSING PREVIEW: A minimal end-to-end example
# ============================================================
# This is a simplified preview of what the rest of Week 4 covers.
# Each step will get an entire notebook later.

from sklearn.preprocessing import LabelEncoder   # converts text categories to numbers
from sklearn.model_selection import train_test_split  # splits data into train/test sets

df_processed = df.copy()

# Step 1: Extract temporal features from the timestamp
# ML models cannot use raw datetime objects — we extract numeric components
df_processed['hour']    = pd.to_datetime(df_processed['timestamp']).dt.hour
df_processed['weekday'] = pd.to_datetime(df_processed['timestamp']).dt.weekday  # 0=Mon, 6=Sun
df_processed['month']   = pd.to_datetime(df_processed['timestamp']).dt.month

# Step 2: Encode the merchant category as a number
# Most ML models require numeric input; LabelEncoder maps each category to an integer
le = LabelEncoder()
df_processed['merchant_category_enc'] = le.fit_transform(df_processed['merchant_category'])

# Step 3: Select features for the model (drop non-numeric and non-feature columns)
feature_cols = ['amount', 'hour', 'weekday', 'month', 'merchant_category_enc']
X = df_processed[feature_cols]  # feature matrix: what the model sees
y = df_processed['is_fraud']     # target vector: what the model predicts

# Step 4: Split into training and test sets
# We train on 80% of data and evaluate on the remaining 20%
# stratify=y ensures both splits have the same fraud rate (important for imbalanced data)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

print('=== PROCESSING SUMMARY ===')
print(f'Original dataset:  {df.shape}  (raw columns: {list(df.columns)})')
print(f'Processed dataset: {df_processed[feature_cols + ["is_fraud"]].shape}')
print(f'Feature matrix X:  {X.shape}')
print(f'\nTrain set: {X_train.shape[0]:,} rows  (fraud: {y_train.sum():,}, rate: {y_train.mean()*100:.1f}%)')
print(f'Test set:  {X_test.shape[0]:,} rows  (fraud: {y_test.sum():,}, rate: {y_test.mean()*100:.1f}%)')
print(f'\nFeatures used: {feature_cols}')
print('\nCategory encoding mapping (first 5):')
for orig, enc in zip(le.classes_[:5], range(5)):
    print(f'  {orig} → {enc}')

---

## Common Data Formats: Read & Write Demo

Before a data scientist can process data, they need to read it. Data arrives in many formats. Here we demonstrate how pandas handles the most common ones.

In [ ]:
# ============================================================
# DATA FORMATS: Reading and writing CSV, JSON, and Parquet
# ============================================================
# We use small 20-row samples for demonstration clarity

sample = df.head(20).copy()
# Convert timestamp to string for JSON compatibility (datetime isn't JSON-serializable natively)
sample['timestamp'] = sample['timestamp'].astype(str)

# ---- CSV: comma-separated values ----
csv_buf = io.StringIO()
sample.to_csv(csv_buf, index=False)  # index=False: don't write row numbers as a column
csv_buf.seek(0)
df_csv = pd.read_csv(csv_buf)
print('CSV preview (first 3 rows):')
print(df_csv.head(3).to_string())

# ---- JSON: records orientation ----
# orient='records' produces [{col: val, ...}, ...] — natural for APIs and web services
json_str = sample.to_json(orient='records', indent=2)
df_json  = pd.read_json(io.StringIO(json_str), orient='records')
print(f'\nJSON shape: {df_json.shape}  |  First record:')
print(json_str[:300], '...')  # show first 300 characters of JSON

# ---- Parquet: columnar binary format ----
parquet_buf = io.BytesIO()
try:
    sample.to_parquet(parquet_buf, index=False)  # requires pyarrow or fastparquet
    parquet_buf.seek(0)
    df_pq = pd.read_parquet(parquet_buf)
    print(f'\nParquet shape: {df_pq.shape}')
    print('Parquet dtypes (schema enforced):')
    print(df_pq.dtypes.to_string())
except Exception as e:
    print(f'\nParquet skipped: {e}  — run: pip install pyarrow')

print('\nAll 3 formats read back to identical DataFrames.')
print('Choose the format based on your use case, not convenience.')

---

## Visualizing the Full ML Data Lifecycle

Now let's create a visual summary of all 5 stages we've covered, showing how data flows from raw collection to model-ready features.

In [ ]:
# ============================================================
# VISUALIZATION: ML Data Lifecycle Flow Diagram
# ============================================================
# We build a custom diagram using matplotlib primitives (boxes + arrows + text).
# This is a common technique when no dedicated diagram library is available.

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.axis('off')  # hide all axis ticks and labels — we're drawing manually

# Define the stages: (x_center, y_center, label, detail, color)
stages = [
    (1.2, 4.0, '1. Collection',  'Sources:\nSQL, APIs,\nSensors',     '#4A90D9'),
    (3.8, 4.0, '2. Labeling',    'Fraud / Legit\nExpert review\nWeak supervision', '#E8894A'),
    (6.4, 4.0, '3. Storage',     'CSV, SQL,\nParquet,\nJSON',         '#5BAD6F'),
    (9.0, 4.0, '4. Versioning',  'Snapshots,\nLineage logs,\nDVC',    '#9B59B6'),
    (11.6, 4.0, '5. Processing', 'Clean → Features\n→ Encode → Scale\n→ Split', '#E74C3C'),
]

box_w, box_h = 2.0, 2.4  # box width and height

for x, y, title, detail, color in stages:
    # Draw the stage box as a rounded rectangle (FancyBboxPatch)
    box = mpatches.FancyBboxPatch(
        (x - box_w/2, y - box_h/2),  # lower-left corner
        box_w, box_h,
        boxstyle='round,pad=0.1',     # rounded corners with padding
        facecolor=color,
        edgecolor='white',
        linewidth=2,
        alpha=0.88
    )
    ax.add_patch(box)
    
    # Stage title (bold white)
    ax.text(x, y + 0.55, title, ha='center', va='center',
            fontsize=10, fontweight='bold', color='white')
    # Stage detail (smaller white text)
    ax.text(x, y - 0.25, detail, ha='center', va='center',
            fontsize=8, color='white', alpha=0.92, linespacing=1.5)

# Draw arrows between stages
arrow_y = 4.0
for x_start, x_end in [(2.2, 2.8), (4.8, 5.4), (7.4, 8.0), (10.0, 10.6)]:
    ax.annotate('', xy=(x_end, arrow_y), xytext=(x_start, arrow_y),
                arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=2.5))

# Draw the "Model" output box at the right end
model_box = mpatches.FancyBboxPatch((12.3, 3.2), 1.4, 1.6,
                                     boxstyle='round,pad=0.1',
                                     facecolor='#2C3E50', edgecolor='#F39C12',
                                     linewidth=2.5)
ax.add_patch(model_box)
ax.text(13.0, 4.0, 'ML\nModel', ha='center', va='center',
        fontsize=11, fontweight='bold', color='#F39C12')
ax.annotate('', xy=(12.3, 4.0), xytext=(12.1, 4.0),
            arrowprops=dict(arrowstyle='->', color='#F39C12', lw=2.5))

# Title and subtitle
ax.text(7.0, 7.3, 'The ML Data Lifecycle',
        ha='center', fontsize=16, fontweight='bold', color='#2C3E50')
ax.text(7.0, 6.8, 'Credit Card Fraud Detection: From Raw Transactions to Model',
        ha='center', fontsize=11, color='#555555', style='italic')

# Bottom annotation
ax.text(7.0, 1.2,
        '"80% of ML time is spent on data (stages 1–4). The model (stage 5 output) is the easy part."',
        ha='center', fontsize=10, color='#777777', style='italic')

plt.tight_layout()
plt.show()

---

## Bringing It All Together: A Full Lifecycle Simulation

In [ ]:
# ============================================================
# FULL LIFECYCLE SIMULATION: From collection to versioned storage
# ============================================================
# This cell runs through all 5 stages in sequence,
# printing a log that mirrors what a real pipeline would output.

def lifecycle_simulation():
    """Simulate the full data lifecycle for a fraud detection dataset."""
    log = []
    ts = lambda: datetime.datetime.now().strftime('%H:%M:%S.%f')[:12]  # short timestamp

    print('=' * 60)
    print('  ML DATA LIFECYCLE SIMULATION — Fraud Detection')
    print('=' * 60)

    # Stage 1: Collection
    print(f'\n[{ts()}] STAGE 1: Data Collection')
    print(f'         Ingesting {N:,} transactions from transaction_log...')
    raw_df = df.copy()  # our pre-generated dataset represents the raw source
    print(f'         Collected: {raw_df.shape[0]:,} rows, {raw_df.shape[1]} columns')
    print(f'         Date range: {raw_df["timestamp"].min()} → {raw_df["timestamp"].max()}')
    log.append({'stage': 'collection', 'rows': len(raw_df), 'status': 'OK'})

    # Stage 2: Labeling
    print(f'\n[{ts()}] STAGE 2: Data Labeling')
    fraud_count = raw_df['is_fraud'].sum()
    print(f'         Labeled: {fraud_count:,} fraud, {len(raw_df)-fraud_count:,} legitimate')
    print(f'         Estimated labeling cost: ${fraud_count * 2:,.0f} (at $2 per fraud label)')
    print(f'         Label quality: assumed 98% (2% noise tolerance)')
    log.append({'stage': 'labeling', 'fraud_labels': int(fraud_count), 'status': 'OK'})

    # Stage 3: Storage (simulate writing to CSV buffer)
    print(f'\n[{ts()}] STAGE 3: Data Storage')
    buf = io.StringIO()
    raw_df['timestamp'] = raw_df['timestamp'].astype(str)  # serialize for CSV
    raw_df.to_csv(buf, index=False)
    size_mb = buf.tell() / (1024 * 1024)  # convert bytes to megabytes
    print(f'         Saved to: transactions_raw_v1.csv  ({size_mb:.2f} MB)')
    print(f'         Also mirrored to: SQLite in-memory DB [table: transactions]')
    log.append({'stage': 'storage', 'size_mb': round(size_mb, 3), 'status': 'OK'})

    # Stage 4: Versioning
    print(f'\n[{ts()}] STAGE 4: Data Versioning')
    version_tag = 'v1.0.0'
    version_date = datetime.datetime.now().strftime('%Y%m%d')
    print(f'         Version tag: {version_tag}  |  Date: {version_date}')
    print(f'         Hash (simulated): sha256:{abs(hash(buf.getvalue()))}')
    print(f'         Lineage: raw_collection → no_transforms_applied')
    log.append({'stage': 'versioning', 'version': version_tag, 'status': 'OK'})

    # Stage 5: Processing preview
    print(f'\n[{ts()}] STAGE 5: Data Processing')
    print(f'         → Extracting temporal features (hour, weekday, month)')
    print(f'         → Encoding merchant_category (8 categories → integers)')
    print(f'         → Splitting: 80% train ({int(N*0.8):,}) / 20% test ({int(N*0.2):,})')
    print(f'         Model-ready features: [amount, hour, weekday, month, merchant_enc]')
    log.append({'stage': 'processing', 'features': 5, 'status': 'COMPLETE'})

    print(f'\n[{ts()}] PIPELINE COMPLETE — {len(log)} stages executed successfully.')
    print('=' * 60)
    return log

# Run the simulation
pipeline_log = lifecycle_simulation()

In [ ]:
# ============================================================
# VISUALIZATION: Transaction Volume & Fraud Rate Over Time
# ============================================================
# This is a key diagnostic chart: it shows whether data collection
# is consistent over time and whether fraud patterns shift seasonally.

df_time = df.copy()
df_time['month_year'] = pd.to_datetime(df_time['timestamp']).dt.to_period('M')  # group by month

# Aggregate by month
monthly = df_time.groupby('month_year').agg(
    total_txns=('is_fraud', 'count'),     # total transactions per month
    fraud_txns=('is_fraud', 'sum'),        # fraud transactions per month
    total_amount=('amount', 'sum')         # total transaction value per month
).reset_index()
monthly['fraud_rate'] = monthly['fraud_txns'] / monthly['total_txns'] * 100  # fraud rate %
monthly['month_str'] = monthly['month_year'].astype(str)  # string labels for x-axis

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)  # sharex: aligned x-axis
fig.suptitle('Transaction Data Collection Quality Over Time\n(Lifecycle Stage 1 Diagnostic)',
             fontsize=13, fontweight='bold')

# Top subplot: transaction volume
axes[0].bar(monthly['month_str'], monthly['total_txns'], color='#4A90D9', alpha=0.8, edgecolor='white')
axes[0].set_ylabel('Transactions per Month')
axes[0].set_title('Volume: Are we collecting data consistently?')
# Add a reference line at the mean volume
mean_vol = monthly['total_txns'].mean()
axes[0].axhline(mean_vol, color='#E74C3C', linestyle='--', lw=1.5, label=f'Mean: {mean_vol:.0f}/month')
axes[0].legend(fontsize=9)

# Bottom subplot: fraud rate over time
axes[1].plot(monthly['month_str'], monthly['fraud_rate'],
             marker='o', linewidth=2, color='#E74C3C', markersize=6)
axes[1].fill_between(range(len(monthly)), monthly['fraud_rate'],
                     alpha=0.15, color='#E74C3C')  # shaded area under the line
axes[1].set_ylabel('Fraud Rate (%)')
axes[1].set_title('Fraud Rate: Is the pattern stable? (concept drift check)')
axes[1].set_xlabel('Month')

# Rotate x-axis labels for readability
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(f'Monthly transaction range: {monthly["total_txns"].min()} – {monthly["total_txns"].max()}')
print(f'Monthly fraud rate range: {monthly["fraud_rate"].min():.1f}% – {monthly["fraud_rate"].max():.1f}%')
print('Stable volume = good data collection. Shifting fraud rate = concept drift risk.')

---

## Key Takeaways

| Stage | What Can Go Wrong | Warning Sign |
|---|---|---|
| **Collection** | Sampling bias, missing populations | Model fails on certain user groups |
| **Labeling** | Label noise, disagreement, cost | High annotation error rate |
| **Storage** | Format mismatch, schema drift | Pandas dtype errors on read |
| **Versioning** | No version control, data overwritten | Can't reproduce a past result |
| **Processing** | Leakage, inconsistent transforms | Train accuracy >> test accuracy |

**The Golden Rules:**
1. Never overwrite raw data
2. Log every transformation with a timestamp
3. Verify your data is representative before modeling
4. Version your datasets like you version your code

---

---

## Self-Check Questions

Test your understanding before moving on. Try answering from memory first, then reveal the answers.

---

**Question 1:** Your fraud model was 95% accurate last year but is now 70% accurate. You haven't changed the model code. Which lifecycle stage is most likely the issue, and why?

<details>
<summary>Click to reveal answer</summary>

**Stage 1 (Collection) or Stage 5 (Processing) — most likely a data freshness/concept drift issue.**

The model hasn't changed, but the world has. The most probable causes are:
- **Data Collection**: The fraud patterns in production have changed (new attack vectors, COVID-era spending shifts), but the training data was not updated to reflect these changes. The model is still pattern-matching against 2023 fraud when 2024 fraud looks different.
- **Distribution shift**: The types of transactions being processed have changed (new merchant categories, new user demographics) that weren't in the training data.

The fix: retrain with fresh, recent data. Monitor the **fraud rate over time** (as shown in the last visualization) — a sudden shift signals concept drift. This is also why versioning matters: you can roll back to the last known-good dataset to compare.

</details>

---

**Question 2:** You have 1 million transactions but only 0.1% are labeled as fraud (1,000 labeled fraud cases). What specific challenge does this create at the **labeling stage**, and what two strategies could help?

<details>
<summary>Click to reveal answer</summary>

**Challenge: Class imbalance and labeling cost.**

With only 0.1% fraud, you have a severe class imbalance — the model can achieve 99.9% accuracy by simply predicting "not fraud" for every transaction. It learns nothing useful about actual fraud.

At the labeling stage, the problem is:
- **Labeling cost**: You'd need to pay experts to review 1 million transactions just to find 1,000 fraud cases. Most of that labeling cost buys you legitimate examples you don't need more of.
- **Underreporting**: Many fraud cases are never reported, so even the 0.1% may be an undercount.

**Two strategies:**
1. **Active learning**: Use the model to identify which unlabeled transactions are most uncertain, then only pay to label those — getting maximum information per dollar spent.
2. **Programmatic labeling / weak supervision**: Write heuristic rules ("charge > $5,000 + new merchant + 3am = likely fraud") to automatically label transactions. This produces noisy labels but at zero cost, and can be combined with the small set of expert-labeled data.

</details>

---

**Question 3:** Why should you NEVER overwrite your raw data file when preprocessing it? Give two specific scenarios where this rule would save you.

<details>
<summary>Click to reveal answer</summary>

**Raw data is the irreplaceable source of truth. Once overwritten, it is gone forever.**

**Scenario 1 — Debugging a bug in preprocessing:**
You applied a normalization step that accidentally capped all transaction amounts at $1,000, discarding data about high-value fraud. If you overwrote the raw file, you cannot go back and fix the capping. With the raw file intact, you simply re-run the corrected preprocessing from scratch.

**Scenario 2 — Reproducing a past result:**
Six months ago your model passed an audit. The regulator now asks you to prove it worked correctly at that time. If you've been overwriting your data, you can only show what the data looks like today — which may have changed. With versioned raw files (e.g., `transactions_raw_2023-01-01.csv`), you can reload the exact dataset the audited model was trained on.

**The rule:** Raw data is append-only. Processing always produces a *new* file. Think of it like lab notebooks — scientists never erase original observations, only add new interpretations.

</details>

In [ ]:
# ============================================================
# BONUS: Visualize the 80/20 Rule — Time Allocation in ML Projects
# ============================================================
# This chart illustrates where ML practitioners actually spend their time.
# Source: Based on surveys of industry ML engineers (Google, Kaggle community reports).

activities = [
    'Data Collection\n& Sourcing',
    'Data Cleaning\n& EDA',
    'Feature\nEngineering',
    'Model Training\n& Tuning',
    'Deployment &\nMonitoring'
]
time_pct = [20, 30, 25, 15, 10]  # approximate percentages from industry surveys
colors   = ['#4A90D9', '#E8894A', '#5BAD6F', '#9B59B6', '#E74C3C']
data_stages = [True, True, True, False, False]  # first 3 are data work, last 2 are model work

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('The 80/20 Rule: Where ML Time Actually Goes', fontsize=13, fontweight='bold')

# Left: Horizontal bar chart
bars = ax1.barh(activities, time_pct, color=colors, edgecolor='white', linewidth=1.5, height=0.6)
for bar, pct in zip(bars, time_pct):
    ax1.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
             f'{pct}%', va='center', fontsize=11, fontweight='bold')
ax1.set_xlabel('Approximate % of Project Time')
ax1.set_xlim(0, 40)
ax1.axvline(0, color='black', lw=0.5)

# Shade the data work region
ax1.axvspan(0, 35, ymin=0.32, ymax=1.0, alpha=0.06, color='#4A90D9', label='Data work (75%)')
ax1.set_title('Time Per Activity')

# Right: Pie chart showing data vs model split
data_time  = sum(p for p, d in zip(time_pct, data_stages) if d)      # 75%
model_time = sum(p for p, d in zip(time_pct, data_stages) if not d)   # 25%
wedges, texts, autotexts = ax2.pie(
    [data_time, model_time],
    labels=['Data Work\n(Collection + Cleaning\n+ Feature Engineering)',
            'Model Work\n(Training + Deployment)'],
    colors=['#4A90D9', '#E74C3C'],
    autopct='%1.0f%%',
    startangle=140,
    pctdistance=0.7,
    textprops={'fontsize': 10}
)
for at in autotexts:
    at.set_fontsize(13)
    at.set_fontweight('bold')
    at.set_color('white')
ax2.set_title('The 80/20 Rule')

plt.tight_layout()
plt.show()

print('Key insight: The entire Week 4 curriculum covers the data work side (the 75%).')
print('Mastering this is what separates a real ML engineer from a notebook practitioner.')

---

## What's Next?

Now that you understand the full ML data lifecycle, the next notebook dives deep into **data quality** — the four dimensions that determine whether your data is actually usable:

- **Completeness** — are all the values there?
- **Consistency** — are the same things stored the same way?
- **Accuracy** — are the values correct (not just present)?
- **Timeliness** — is the data fresh enough to be useful?

**→ Continue to: `02_Data_Quality_Dimensions.ipynb`**

---
*Week 4 · Data Fundamentals & Preprocessing Pipelines*